# ATOM-SLM — train a 135M math SLM on a free Kaggle T4

End-to-end notebook for **M1** of the
[project plan](../docs/Project%20ATOM-SLM%20-%20Plan.md):
fine-tune `SmolLM2-135M` on a MetaMathQA subset, then measure GSM8K before vs.
after.

**Before you run**

1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
2. Add-ons → Secrets: `HF_TOKEN` (Hugging Face **write** token), optionally `WANDB_API_KEY`.
3. Run all cells. Checkpoints are pushed to the Hub every `SAVE_STEPS`, so a
   dead session never costs more than a few hundred steps.

> **T4 rules:** fp16 (never bf16), SDPA attention (not FlashAttention-2),
> gradient checkpointing on, effective batch 64 via gradient accumulation.
>
> This notebook runs on **one** GPU. The second T4 is left free; DDP is not
> worth the complexity for a 135M model.

## 1 — Environment check & dependencies

In [ ]:
import subprocess
import sys

import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPUs   :", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        major, minor = torch.cuda.get_device_capability(i)
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  [{i}] {name} sm_{major}{minor} | {vram:.1f} GB")
    print("bf16 supported:", major >= 8, "(T4 = False -> fp16 + GradScaler)")

INSTALL = True  # set False after the first run in a session

if INSTALL:
    subprocess.run(
        'pip install -q -U "transformers>=4.46" "trl>=0.13" "peft>=0.13" '
        '"accelerate>=1.0" "datasets>=3.0" "huggingface-hub>=0.24" '
        'sentencepiece protobuf "bitsandbytes>=0.44" "pyyaml>=6.0"',
        shell=True,
        check=True,
    )
    print("\nDependencies installed (torch left untouched).")

import datasets
import peft
import transformers
import trl

print(f"transformers {transformers.__version__} | trl {trl.__version__} | "
      f"peft {peft.__version__} | datasets {datasets.__version__}")

## 2 — Get the code

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/asaadzx/Atom_Slm.git"

candidates = [Path.cwd(), Path("/kaggle/working/Atom_Slm"), Path("/kaggle/working")]
REPO_DIR = next((p for p in candidates if (p / "src" / "atom_slm").exists()), None)

if REPO_DIR is None:
    REPO_DIR = Path("/kaggle/working/Atom_Slm")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

SRC_DIR = REPO_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

ENV = {
    **os.environ,
    "PYTHONPATH": f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}",
    "TOKENIZERS_PARALLELISM": "false",
}


def run(cmd: str, cwd: Path = REPO_DIR) -> None:
    """Run a command with the repo on PYTHONPATH and stream its output."""
    print(f"$ {cmd}\n", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd), env=ENV)


import atom_slm

print("Repo   :", REPO_DIR)
print("Package:", atom_slm.__version__)

## 3 — Run configuration

Start with M1 defaults. For **M2** switch `DATA = "openmathinstruct"` and
`MAX_SAMPLES = 500_000`; for **M3** set `BASE_MODEL = "HuggingFaceTB/SmolLM2-360M"`
and `METHOD = "lora"`.

In [ ]:
from pathlib import Path

import yaml

# ---------------------------- experiment knobs ----------------------------
RUN_NAME         = "m1-smollm2-135m-metamath"
BASE_MODEL       = "HuggingFaceTB/SmolLM2-135M"
METHOD           = "full"          # "full" (135M fits easily) or "lora"
HF_REPO_ID       = "Asaadzx/atom-slm-math-135m"
PUSH_TO_HUB      = True

DATA             = "metamath"      # "metamath" or "openmathinstruct"
MAX_SAMPLES      = 60_000
VAL_SAMPLES      = 1_000
MAX_LENGTH       = 1024
EPOCHS           = 1.0
PER_DEVICE_BATCH = 8
GRAD_ACCUM       = 8
LEARNING_RATE    = 5e-5
SAVE_STEPS       = 500
EVAL_STEPS       = 500
# -------------------------------------------------------------------------

RUN_DIR = Path("/kaggle/working/runs") / RUN_NAME
CONFIG_PATH = Path("/kaggle/working") / f"{RUN_NAME}.yaml"

config = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "method": METHOD,
    "precision": "fp16",  # T4/P100 have no bf16
    "output_dir": str(RUN_DIR),
    "seed": 42,
    "data": {
        "name": DATA,
        "max_samples": MAX_SAMPLES,
        "val_samples": VAL_SAMPLES,
        "streaming": True,
    },
    "training": {
        "max_length": MAX_LENGTH,
        "epochs": EPOCHS,
        "per_device_batch_size": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler": "cosine",
        "warmup_ratio": 0.03,
        "optim": "adamw_torch",
        "gradient_checkpointing": True,
        "packing": True,
        "logging_steps": 20,
        "save_steps": SAVE_STEPS,
        "eval_steps": EVAL_STEPS,
        "save_total_limit": 2,
    },
    "hub": {
        "push_to_hub": PUSH_TO_HUB,
        "repo_id": HF_REPO_ID,
        "private": False,
        "strategy": "every_save",
    },
}

if METHOD == "lora":
    config["lora"] = {
        "r": 32,
        "alpha": 64,
        "dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    }

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(f"Config written to {CONFIG_PATH}\n")
print(CONFIG_PATH.read_text())
print(f"Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")

## 4 — Hugging Face login

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    try:
        os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    except Exception:
        pass
except Exception as exc:
    print("Kaggle secrets unavailable:", exc)

from huggingface_hub import login, whoami

if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Logged in as:", whoami()["name"])
else:
    print("No HF_TOKEN: training still works, but checkpoints will NOT be persisted.")

## 5 — Baseline GSM8K (optional, recommended)

Know the starting point before spending GPU time. 200 examples is enough to
see the ~1% base accuracy; use `EVAL_LIMIT = 0` for the full 1319-item test set.

In [ ]:
RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_BASELINE_EVAL = False   # ~10–20 min on T4 for 200 examples
EVAL_LIMIT = 200            # 0 = full GSM8K test set
BASE_EVAL_PATH = RESULTS_DIR / "base-gsm8k.json"

if RUN_BASELINE_EVAL:
    run(
        f"{sys.executable} -m atom_slm.evaluate "
        f"--model {BASE_MODEL} --limit {EVAL_LIMIT} --output {BASE_EVAL_PATH}"
    )
else:
    print("Skipped baseline eval. Set RUN_BASELINE_EVAL = True to run it.")

## 6 — Resume from the Hub (only after a session restart)

Kaggle wipes `/kaggle/working` between sessions. After a timeout, set
`RESUME_FROM_HUB = True` and rerun: checkpoints are pulled back, then
`train` resumes from them automatically (`--resume auto`).

In [ ]:
RESUME_FROM_HUB = False

if RESUME_FROM_HUB:
    from huggingface_hub import snapshot_download

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_REPO_ID,
        local_dir=str(RUN_DIR),
        allow_patterns=["checkpoint-*/**"],
    )
    checkpoints = sorted(RUN_DIR.glob("checkpoint-*"))
    print("Restored checkpoints:", [p.name for p in checkpoints] or "none")
else:
    print("No Hub restore requested.")

## 7 — Train

This runs `atom_slm.train` as a subprocess so logs stream live and the notebook
stays small. Expect roughly **1–3 h** for 60k rows on a T4.

What the trainer does: format MetaMathQA into the SmolLM2 chat template, enforce
`#### <answer>`, pack sequences, train with fp16 + GradScaler, log to the Hub
every `SAVE_STEPS`.

In [ ]:
extra = "" if PUSH_TO_HUB else " --no-push"
run(
    f"{sys.executable} -m atom_slm.train "
    f"--config {CONFIG_PATH} --output-dir {RUN_DIR}{extra}"
)

## 8 — Evaluate the tuned model and compare

In [ ]:
TUNED_EVAL_PATH = RESULTS_DIR / f"{RUN_NAME}-gsm8k.json"

run(
    f"{sys.executable} -m atom_slm.evaluate "
    f"--model {RUN_DIR} --limit {EVAL_LIMIT} --output {TUNED_EVAL_PATH}"
)

reports = [str(p) for p in (BASE_EVAL_PATH, TUNED_EVAL_PATH) if p.exists()]
if reports:
    run(f"{sys.executable} -m atom_slm.evaluate --compare {' '.join(reports)}")
else:
    print("No reports yet — run the baseline and/or tuned eval first.")

## 9 — Artifacts

In [ ]:
from atom_slm.utils import human_bytes

files = sorted(p for p in RUN_DIR.rglob("*") if p.is_file())
print(f"Run directory: {RUN_DIR}\n")
for path in files[:40]:
    print(f"{human_bytes(path.stat().st_size):>10}  {path.relative_to(RUN_DIR)}")
print(f"{'':>10}  ----")
print(f"{human_bytes(sum(p.stat().st_size for p in files)):>10}  total\n")

metrics = RUN_DIR / "metrics.json"
if metrics.exists():
    print(metrics.read_text())
if PUSH_TO_HUB:
    print(f"Model: https://huggingface.co/{HF_REPO_ID}")

## 10 — Optional: merge + export to GGUF (M4)

GGUF conversion needs a llama.cpp checkout, which is CPU work. Set
`EXPORT_GGUF = True` after a successful training run. Quantized files
(`Q8_0`, `Q5_K_M`, `Q4_K_M`) land in the working directory — download them from
the notebook output before the session ends.

In [ ]:
EXPORT_GGUF = False

if EXPORT_GGUF:
    merged = Path("/kaggle/working/merged")
    run(
        f"{sys.executable} -m atom_slm.merge "
        f"--base {BASE_MODEL} --adapter {RUN_DIR} --output {merged}"
    )
    run(f"bash scripts/convert_gguf.sh {merged} atom-slm-math")
else:
    print("Skipped GGUF export (set EXPORT_GGUF = True after training).")

## Troubleshooting

| Symptom | Fix |
|---|---|
| CUDA OOM | halve `PER_DEVICE_BATCH`, keep effective batch via `GRAD_ACCUM`; lower `MAX_LENGTH` |
| `bf16` NaNs / garbage | never force bf16 on T4 — `precision: fp16` |
| Loss flat from step 0 | wrong chat template or LR; check `tokenizer.chat_template` and the printed sample |
| Run killed at ~9–12 h | expected: new session → `RESUME_FROM_HUB = True` → rerun from section 6 |
| Hub push fails | `HF_TOKEN` missing/read-only, or repo id does not match your account |
| Disk full | `save_total_limit: 1`, rely on Hub uploads |

**Next milestones:** M2 scales the data (`DATA = "openmathinstruct"`), M3 compares
SmolLM2-360M with LoRA, M4 converts to GGUF. See `docs/Project ATOM-SLM - Plan.md`
and `docs/kaggle.md` in the repo.